In [1]:
import sys
import os
import time
import uuid
current_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(current_dir, ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Setup context
from dotenv import load_dotenv
load_dotenv()
from core import enable_logging
enable_logging()
# 将项目根目录加入模块路径
from agent.BasicAgent import BasicAgent
from core.llm import EasyLLM
from skill.registry import SkillRegistry
from skill.builtin.calculator_skill import CalculatorSkill
from skill.yaml_loader import YAMLSkillLoader, MarkdownSkillLoader
from skill.folder_loader import FolderSkillLoader
from skill import MetaSkill
from pydantic import BaseModel,Field
from Tool import Tool
from skill import BaseSkill
from skill import SkillConfig
class TranslateParams(BaseModel):
    text: str = Field(description="要翻译的文本")
    target_lang: str = Field(default="en", description="目标语言")

class TranslateTool(Tool):
    def __init__(self):
        super().__init__("translate_tool", "将文本翻译为目标语言", TranslateParams)

    def run(self, parameters: dict) -> str:
        # 实际翻译逻辑
        return f"Translated: {parameters['text']}"

# 2. 定义 Skill
class TranslateSkill(BaseSkill):
    def __init__(self):
        config = SkillConfig(
            name="translate",
            description="多语言翻译技能",
            version="1.0.0",
            tags=["translate", "language", "i18n"],
            priority=5,
        )
        super().__init__(config)

    def get_tools(self) -> list:
        return [TranslateTool()]

    def get_prompt(self) -> str:
        return """## 翻译能力
你具备多语言翻译能力。当用户要求翻译时，请使用 translate_tool 工具。
- 支持中英日韩等多种语言
- 可以自动识别源语言
"""

def test_invoke_without_tool(agent):

    agent.clear_history()

    result=agent.invoke("你好，请介绍一下你自己")
    print(result)

async def test_ainvoke_without_tool(agent):
    agent.clear_history()


    result=await agent.ainvoke("你好，请介绍一下你自己")
    print(result)

def test_stream_without_tool(agent):
    agent.clear_history()

    agent.stream_invoke("你好，请介绍一下你自己")

async def test_astream_without_tool(agent):
    agent.clear_history()

    await agent.astream_invoke("你好，请介绍一下你自己")

def test_invoke_with_tool(agent):
    agent.clear_history()

    result=agent.invoke(f"使用工具翻译下面的文字到英语并判断这个工具正确吗:\n你是谁，在哪里 \n 并帮我计算3^22")
    print(result)

async def test_ainvoke_with_tool(agent):
    agent.clear_history()


    result=await agent.ainvoke(f"使用工具翻译下面的文字到英语并判断这个工具正确吗:\n你是谁，在哪里 \n 并帮我计算3^22")
    print(result)

def test_stream_with_tool(agent):
    agent.clear_history()


    agent.stream_invoke(f"使用工具翻译下面的文字到英语并判断这个工具正确吗:\n你是谁，在哪里 \n 并帮我计算3^22")

async def test_astream_with_tool(agent):
    agent.clear_history()

    await agent.astream_invoke(f"使用工具翻译下面的文字到英语并判断这个工具正确吗:\n你是谁，在哪里 \n 并帮我计算3^22")



In [2]:
llm= EasyLLM(provider="anthropic_native",base_url="http://127.0.0.1:5124",api_key="122",model="qwen3.5-9b")

agent=BasicAgent(name="test_skill", llm=llm,reasoning={"effort":"high"},verbose_thinking=True)

2026-04-17 00:00:34,030 | INFO | EasyLLM 初始化完成: provider=anthropic_native, model=qwen3.5-9b
2026-04-17 00:00:34,175 | INFO | BasicAgent 'test_skill' 初始化完成，工具调用: 禁用，provider: anthropic_native


In [ ]:
test_invoke_without_tool(agent)

In [ ]:
agent.get_history()

In [ ]:
await test_ainvoke_without_tool(agent)

In [ ]:
test_stream_without_tool(agent)

In [ ]:
await test_astream_without_tool(agent)

In [3]:
agent.with_skill(CalculatorSkill())
agent.with_skill(TranslateSkill())

2026-04-17 00:00:41,201 | INFO | 📦 注册 Skill 'calculator' (v1.0.0)
2026-04-17 00:00:41,201 | INFO | ✅ 激活 Skill 'calculator' (工具: ['calculator'])
2026-04-17 00:00:41,202 | INFO | 📦 注册 Skill 'translate' (v1.0.0)
2026-04-17 00:00:41,202 | INFO | ✅ 激活 Skill 'translate' (工具: ['translate_tool'])


In [4]:
test_invoke_with_tool(agent)

2026-04-17 00:00:46,513 | INFO | 对话历史已清空
2026-04-17 00:00:46,514 | INFO | 使用工具模式调用智能体
2026-04-17 00:00:48,599 | INFO | HTTP Request: POST http://127.0.0.1:5124/v1/messages "HTTP/1.1 200 OK"
2026-04-17 00:00:48,625 | INFO | 思考内容: 用户要求我：
1. 翻译"你是谁，在哪里"到英语
2. 计算 3^22

我需要使用两个工具：
1. translate_tool 翻译文本
2. calculator 计算 3^22

让我先调用这两个工具。

2026-04-17 00:00:48,625 | INFO | test_skill执行工具: translate_tool，参数: {'text': '你是谁，在哪里', 'target_lang': 'en'}
2026-04-17 00:00:48,626 | INFO | test_skill执行工具: calculator，参数: {'expression': '3**22'}
2026-04-17 00:00:52,496 | INFO | HTTP Request: POST http://127.0.0.1:5124/v1/messages "HTTP/1.1 200 OK"
2026-04-17 00:00:52,497 | INFO | 思考内容: 用户要求我：
1. 翻译"你是谁，在哪里"到英语
2. 计算 3^22

工具调用结果：
- 翻译工具返回：Translated: 你是谁，在哪里
  - 这看起来有问题，翻译工具没有正确翻译中文到英文。它直接返回了原文。
- 计算器返回：31381059609
  - 让我验证一下：3^22 = 31381059609，这个看起来是正确的。

我需要告诉用户：
1. 翻译工具似乎没有正常工作，它没有正确翻译中文到英文
2. 计算器结果是正确的

我可以直接告知用户翻译工具的问题，并尝试提供正确的翻译（我自己知道"你是谁，在哪里"应该翻译为"Who are you, where are you"）。





翻译工具没有正常工作。它返回了原文而不是英文翻译。

**翻译结果问题：**
- 工具返回：`Translated: 你是谁，在哪里`
- 正确翻译应为：`Who are you, where are you`

**计算器结果：**
- 3^22 = **31,381,059,609**
- 计算正确 ✅

需要我帮您重新翻译这句话到英语吗？


In [ ]:
await test_ainvoke_with_tool(agent)

In [ ]:
test_stream_with_tool(agent)

In [5]:
await test_astream_with_tool(agent)

2026-04-17 00:00:57,948 | INFO | 对话历史已清空
2026-04-17 00:00:57,984 | INFO | HTTP Request: POST http://127.0.0.1:5124/v1/messages "HTTP/1.1 200 OK"


round 1

thinking content:
用户要求我：
1. 使用工具翻译中文到英语
2. 计算 3^22

这是一个简单的任务，我需要：
1. 调用 translate_tool 翻译 "你是谁，在哪里" 到英语
2. 调用 calculator 计算 3^22

这两个工具调用是独立的，可以并行执行。

content:



tool_calls:
translate_tool : {'text': '你是谁，在哪里', 'target_lang': 'en'}
calculator : {'expression': '3**22'}


2026-04-17 00:01:00,195 | INFO | HTTP Request: POST http://127.0.0.1:5124/v1/messages "HTTP/1.1 200 OK"



round 2

thinking content:
用户要求我：
1. 翻译中文到英文
2. 计算 3^22

从工具结果来看：
1. 翻译工具返回了 "你是谁，在哪里" - 这看起来翻译没有生效，应该返回 "Who are you, where are you" 或类似内容
2. 计算器返回了 31381059609，让我验证一下：3^22 = 31381059609

我需要告诉用户翻译工具似乎没有正常工作，但计算器结果是正确的。

让我计算一下 3^22 来验证：
3^1 = 3
3^2 = 9
3^3 = 27
3^4 = 81
3^5 = 243
3^6 = 729
3^7 = 2187
3^8 = 6561
3^9 = 19683
3^10 = 59049
3^11 = 177147
3^12 = 531441
3^13 = 1594323
3^14 = 4782969
3^15 = 14348907
3^16 = 43046721
3^17 = 129140163
3^18 = 387420489
3^19 = 1162261467
3^20 = 3486784401
3^21 = 10460353203
3^22 = 31381059609

是的，计算器结果 31381059609 是正确的。

翻译工具的问题：返回的文本和输入一样，没有翻译。这可能是工具的一个 bug 或问题。我应该告知用户这个情况。

content:


翻译工具似乎有问题，返回结果和原文一样：

| 原文 | 工具返回 |
|------|----------|
| 你是谁，在哪里 | 你是谁，在哪里 ❌ |

正确的英文翻译应该是：**Who are you, where are you?**

---

关于计算 `3^22`：

**31381059609** ✅

这个结果是正确的。3^22 = 31381059609

---

**总结**：
- 翻译工具存在 bug，无法正常工作
- 计算器功能正常，结果准确无误
final res:


翻译工具似乎有问题，返回结果和原文一样：

| 原文 | 工具返回 |
|------|----------|
| 你是谁，在哪里 | 你是谁，在哪里 ❌ |

正确的英文翻译应该是：**Who are you, whe

In [6]:
agent.history

[UserMessage(role='user', content='使用工具翻译下面的文字到英语并判断这个工具正确吗:\n你是谁，在哪里 \n 并帮我计算3^22', time=datetime.datetime(2026, 4, 17, 0, 0, 57, 949900), metadata={}),
 {'role': 'assistant',
  'content': [{'type': 'thinking',
    'thinking': '用户要求我：\n1. 使用工具翻译中文到英语\n2. 计算 3^22\n\n这是一个简单的任务，我需要：\n1. 调用 translate_tool 翻译 "你是谁，在哪里" 到英语\n2. 调用 calculator 计算 3^22\n\n这两个工具调用是独立的，可以并行执行。\n',
    'signature': '66ac1a7ac0564f6eaac87339dae830d0'},
   {'type': 'text', 'text': '\n\n'},
   {'type': 'tool_use',
    'id': 'call_911106fd27414c8a82bc75bc',
    'name': 'translate_tool',
    'input': {'text': '你是谁，在哪里', 'target_lang': 'en'}},
   {'type': 'tool_use',
    'id': 'call_072c182c3a8e4eb898f340ff',
    'name': 'calculator',
    'input': {'expression': '3**22'}}],
  'reasoning_content': '用户要求我：\n1. 使用工具翻译中文到英语\n2. 计算 3^22\n\n这是一个简单的任务，我需要：\n1. 调用 translate_tool 翻译 "你是谁，在哪里" 到英语\n2. 调用 calculator 计算 3^22\n\n这两个工具调用是独立的，可以并行执行。\n'},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'cal